# MLOps Group Project — Emotion Classification (Kaggle Training)
Group 2 · IIT Jodhpur · Nikhil Saini · Sharathchandrika · Sarthak Kapoor · Aryaveer Rathi

**Before running:**
1. Settings → Accelerator → **GPU T4**
2. Settings → **Internet ON** (critical — Secrets won't load without this)
3. Add-ons → **Secrets** → add `WANDB_API_KEY` and `HF_TOKEN`

Run this notebook **twice** (once per version). Change the `CONFIG` cell between runs.
- **v1:** `microsoft/MiniLM-L12-H384-uncased`, lr 3e-5, epochs 3
- **v2:** `google/electra-small-discriminator`, lr 5e-5, epochs 4

In [ ]:
!pip -q install -U transformers datasets wandb huggingface_hub scikit-learn

## Step 1 — Load secrets (never hardcode tokens)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
# .strip() is critical — trailing whitespace breaks HF login
HF_TOKEN = secrets.get_secret('HF_TOKEN').strip()
login(token=HF_TOKEN)
wandb.login()
print('Secrets loaded.')

## Step 2 — CONFIG (change this cell between v1 and v2)

In [ ]:
CONFIG = {
    'model_name':    'microsoft/MiniLM-L12-H384-uncased',  # v2: google/electra-small-discriminator
    'version':       'v1',                                  # v2: 'v2'
    'epochs':        3,                                     # v2: 4
    'batch_size':    16,
    'learning_rate': 3e-5,                                  # v2: 5e-5
    'max_length':    128,
}
CONFIG

## Step 3 — Data + id2label

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset('dair-ai/emotion')
labels = ds['train'].features['label'].names
id2label = {i: n for i, n in enumerate(labels)}
label2id = {n: i for i, n in enumerate(labels)}
num_labels = len(labels)
print(id2label)

tok = AutoTokenizer.from_pretrained(CONFIG['model_name'])
def tokenize(b):
    return tok(b['text'], truncation=True, padding='max_length', max_length=CONFIG['max_length'])
ds = ds.map(tokenize, batched=True).rename_column('label', 'labels')
ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

## Step 4 — Model + W&B init

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'], num_labels=num_labels,
    id2label=id2label, label2id=label2id)

# Guard: finish any stale run before starting fresh (prevents duplicate/confused logging)
if wandb.run is not None:
    wandb.finish()

# Fresh project to avoid broken panels from old runs
wandb.init(project='mlops-groupproject-v2', name=f"run-{CONFIG['version']}", config=CONFIG)

# Fix W&B chart axes: define step metrics so eval charts plot against epoch
wandb.run.define_metric('epoch', hidden=True)
wandb.run.define_metric('train/*', step_metric='train/global_step')
wandb.run.define_metric('eval/*', step_metric='epoch')
print('W&B run started — metrics configured for proper chart axes.')

## Step 5 — Train (report_to='wandb')

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='weighted')}

args = TrainingArguments(
    output_dir=f"./results-{CONFIG['version']}",
    num_train_epochs=CONFIG['epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    learning_rate=CONFIG['learning_rate'],
    logging_strategy='steps',        # <-- stream training history as time-series
    logging_steps=20,                # <-- frequent points = visible curves
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='wandb',
    run_name=f"run-{CONFIG['version']}")

trainer = Trainer(model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['validation'],
    compute_metrics=compute_metrics)
trainer.train()

## Step 6 — Test metrics → W&B summary

In [ ]:
test_metrics = trainer.evaluate(ds['test'])
print('TEST:', test_metrics)
wandb.run.summary.update(test_metrics)

## Step 7 — Push BEST model to Hugging Face (run only for the winning version)
Set the HF repo to **Public** afterwards.

In [ ]:
# Push ONLY the best model (run this cell for the winning version only)
# v1 repo: 'Nikhil-iitj/emotion-minilm'  |  v2 repo: 'Nikhil-iitj/emotion-electra'
HF_REPO = 'Nikhil-iitj/emotion-minilm'  # <-- change if v2 wins

trainer.save_model('best'); tok.save_pretrained('best')

import os
size_mb = os.path.getsize('best/model.safetensors') / 1e6
print(f'Model size: {size_mb:.1f} MB')   # confirm < 200 MB for the report

model.push_to_hub(HF_REPO)
tok.push_to_hub(HF_REPO)
wandb.run.summary['huggingface_model'] = f'https://huggingface.co/{HF_REPO}'
wandb.finish()
print(f'Model pushed to https://huggingface.co/{HF_REPO}')